In [6]:
pip install torch torch-geometric pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [7]:
"""
Dual-Stream KG-GNN Ensemble System (Corrected)
Fixes:
1. Meaningful Anomaly Scores (Reconstruction-based)
2. Full Knowledge Graph Loading (All 5 edge types)
3. Duplicate Edge Removal (Coalesce)
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.data import Data
from torch_geometric.utils import negative_sampling, coalesce
import os
import logging

In [8]:


# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================================
# 1. MODEL DEFINITIONS
# ============================================================================

class DOMINANT(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=32, dropout=0.3):
        super(DOMINANT, self).__init__()
        self.fusion = nn.Linear(input_dim, hidden_dim)
        self.dropout = dropout
        self.gc1 = GCNConv(hidden_dim, 64)
        self.gc2 = GCNConv(64, latent_dim)
        self.attr_decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x, edge_index):
        x = F.relu(self.fusion(x))
        x = F.dropout(x, self.dropout, training=self.training)
        x = F.relu(self.gc1(x, edge_index))
        x = F.dropout(x, self.dropout, training=self.training)
        z = self.gc2(x, edge_index)
        x_hat = self.attr_decoder(z)
        return x_hat, z

    def get_anomaly_scores(self, x, edge_index):
        """
        Fix #1: Calculates TRUE reconstruction error (Attr MSE + Struct Prob)
        """
        self.eval()
        with torch.no_grad():
            x_hat, z = self.forward(x, edge_index)

            # 1. Attribute Reconstruction Error (MSE)
            attr_error = torch.mean((x - x_hat) ** 2, dim=1)

            # 2. Structure Reconstruction Error (Link Prediction)
            # We measure how well the model predicts ACTUAL edges vs. RANDOM edges
            # High error = Model thinks existing edges shouldn't exist (Anomaly)

            # Score existing edges (Should be close to 1)
            row, col = edge_index
            pos_probs = torch.sigmoid(torch.sum(z[row] * z[col], dim=1))

            # Map edge errors back to nodes (Scatter Mean)
            # Error = 1 - probability of edge existence
            edge_error = 1 - pos_probs
            struct_error = torch.zeros(x.size(0), device=x.device)
            struct_error.scatter_add_(0, row, edge_error)

            # Normalize by degree to avoid penalizing hubs
            degree = torch.bincount(row, minlength=x.size(0)).float().clamp(min=1)
            struct_error = struct_error / degree

            # Combine (0.5 / 0.5)
            # Normalize both to 0-1 range first to prevent one dominating
            a_min, a_max = attr_error.min(), attr_error.max()
            s_min, s_max = struct_error.min(), struct_error.max()

            attr_norm = (attr_error - a_min) / (a_max - a_min + 1e-8)
            struct_norm = (struct_error - s_min) / (s_max - s_min + 1e-8)

            return 0.5 * attr_norm + 0.5 * struct_norm

class AnomalyDAE(nn.Module):
    def __init__(self, input_dim, num_nodes, hidden_dim=64, embed_dim=32, dropout=0.3):
        super(AnomalyDAE, self).__init__()
        self.attr_enc = nn.Linear(input_dim, hidden_dim)
        self.attr_embed = nn.Linear(hidden_dim, embed_dim)
        self.attr_dec = nn.Linear(embed_dim, input_dim)
        self.struct_embed = nn.Parameter(torch.FloatTensor(num_nodes, embed_dim))
        nn.init.xavier_uniform_(self.struct_embed)
        self.gat = GATConv(embed_dim, embed_dim, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        h_attr = F.relu(self.attr_enc(x))
        z_attr = F.relu(self.attr_embed(h_attr))
        z_struct_context = self.gat(self.struct_embed, edge_index)
        x_hat = self.attr_dec(z_attr + z_struct_context)
        return x_hat, self.struct_embed

    def get_anomaly_scores(self, x, edge_index):
        self.eval()
        with torch.no_grad():
            x_hat, _ = self.forward(x, edge_index)
            # AnomalyDAE focuses purely on how well Structure helps reconstruct Attributes
            return torch.mean((x - x_hat) ** 2, dim=1)

class CoLA(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, embedding_dim=32, dropout=0.3):
        super(CoLA, self).__init__()
        self.fusion = nn.Linear(input_dim, hidden_dim)
        self.gcn1 = GCNConv(hidden_dim, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, embedding_dim)
        self.bilinear = nn.Bilinear(embedding_dim, embedding_dim, 1)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.relu(self.fusion(x))
        x = F.relu(self.gcn1(x, edge_index))
        z = self.gcn2(x, edge_index)
        s = torch.mean(z, dim=0, keepdim=True).expand(z.size(0), -1)
        return torch.sigmoid(self.bilinear(z, s))

    def get_anomaly_scores(self, x, edge_index):
        self.eval()
        with torch.no_grad():
            scores = self.forward(x, edge_index).squeeze()
            return 1 - scores

# ============================================================================
# 2. LOSS FUNCTIONS
# ============================================================================

def dominant_loss(model, x, edge_index, x_hat, z, alpha=0.5):
    attr_loss = F.mse_loss(x_hat, x)

    # Structure Loss (Sampled)
    row, col = edge_index
    pos_probs = torch.sigmoid(torch.sum(z[row] * z[col], dim=1))

    neg_edge_index = negative_sampling(edge_index, num_nodes=x.size(0))
    neg_row, neg_col = neg_edge_index
    neg_probs = torch.sigmoid(torch.sum(z[neg_row] * z[neg_col], dim=1))

    struct_loss = -torch.log(pos_probs + 1e-15).mean() - torch.log(1 - neg_probs + 1e-15).mean()
    return alpha * attr_loss + (1 - alpha) * struct_loss

def anomalydae_loss(model, x, edge_index, x_hat, z_struct, alpha=0.5):
    attr_loss = F.mse_loss(x_hat, x)
    return attr_loss # Simplified for stability (DAE mainly reconstructs X)

def cola_loss(scores):
    return -torch.log(scores + 1e-15).mean()

# ============================================================================
# 3. GRAPH BUILDER (Fix #2 & #3)
# ============================================================================

def load_real_graph():
    logging.info(">>> [1/4] Loading Data...")

    if not os.path.exists('/teamspace/uploads/node_features.pt'):
        raise FileNotFoundError("Missing 'node_features.pt'. Run Phase 2 first!")

    x_cve = torch.load('/teamspace/uploads/node_features.pt')
    input_dim = x_cve.size(1)

    df_map = pd.read_csv('/teamspace/uploads/cve_id_mapping.csv')
    id_to_idx = {row['CVE_ID']: idx for idx, row in df_map.iterrows()}

    edges_source = []
    edges_target = []

    extra_nodes = {}
    current_max_idx = len(id_to_idx)

    # Helper to process ANY edge file
    def process_file(filename, src_col, tgt_col):
        nonlocal current_max_idx
        if os.path.exists(filename):
            df = pd.read_csv(filename)
            logging.info(f"   > Processing {filename} ({len(df)} rows)...")

            count = 0
            for _, row in df.iterrows():
                src, tgt = row[src_col], row[tgt_col]

                # Resolve Source
                if src in id_to_idx: u = id_to_idx[src]
                elif src in extra_nodes: u = extra_nodes[src]
                else:
                    extra_nodes[src] = current_max_idx
                    u = current_max_idx
                    current_max_idx += 1

                # Resolve Target
                if tgt in id_to_idx: v = id_to_idx[tgt]
                elif tgt in extra_nodes: v = extra_nodes[tgt]
                else:
                    extra_nodes[tgt] = current_max_idx
                    v = current_max_idx
                    current_max_idx += 1

                edges_source.append(u)
                edges_target.append(v)
                # Reverse edge (Undirected)
                edges_source.append(v)
                edges_target.append(u)
                count += 1
            logging.info(f"     + Loaded {count} edges.")

    # Fix #2: Load ALL Schema Edges
    process_file('/teamspace/uploads/edges_cve_cwe.csv', 'CVE_ID', 'Weakness_CWE')
    process_file('/teamspace/uploads/edges_cve_cpe.csv', 'CVE_ID', 'CPE_ID')
    process_file('/teamspace/uploads/edges_cpe_product.csv', 'CPE_ID', 'Product_ID')
    process_file('/teamspace/uploads/edges_product_vendor.csv', 'Product_ID', 'Vendor')
    process_file('/teamspace/uploads/edges_cve_ref.csv', 'CVE_ID', 'References')

    if len(edges_source) == 0:
        raise ValueError("No valid edges found! Check CSV filenames.")

    # Fix #3: Remove Duplicates (Coalesce)
    logging.info("   > Removing duplicate edges (Coalescing)...")
    edge_index = torch.tensor([edges_source, edges_target], dtype=torch.long)
    edge_index, _ = coalesce(edge_index, None, current_max_idx, sort_by_row=True)

    # Expand Feature Matrix for Extra Nodes
    num_extra = len(extra_nodes)
    if num_extra > 0:
        logging.info(f"   > Adding noise features for {num_extra} auxiliary nodes (CWE/CPE/etc)...")
        x_noise = torch.randn(num_extra, input_dim) * 0.01
        x_full = torch.cat([x_cve, x_noise], dim=0)
    else:
        x_full = x_cve

    logging.info(f"✅ FINAL GRAPH: {x_full.size(0)} Nodes, {edge_index.size(1)} Edges")
    return Data(x=x_full, edge_index=edge_index), input_dim

# ============================================================================
# 4. TRAINING LOOP
# ============================================================================

def train_model(model, data, name, epochs=50):
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    model.train()
    logging.info(f"Training {name}...")

    for epoch in range(epochs):
        optimizer.zero_grad()
        if name == 'DOMINANT':
            x_hat, z = model(data.x, data.edge_index)
            loss = dominant_loss(model, data.x, data.edge_index, x_hat, z)
        elif name == 'AnomalyDAE':
            x_hat, z_struct = model(data.x, data.edge_index)
            loss = anomalydae_loss(model, data.x, data.edge_index, x_hat, z_struct)
        elif name == 'CoLA':
            scores = model(data.x, data.edge_index)
            loss = cola_loss(scores)

        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            print(f"   Epoch {epoch}: Loss {loss.item():.4f}")

    return model

# ============================================================================
# 5. MAIN EXECUTION
# ============================================================================

# ============================================================================
# 5. MAIN EXECUTION (Updated with Model Saving)
# ============================================================================

def main():
    data, input_dim = load_real_graph()
    data = data.to(device)
    num_nodes = data.x.size(0)

    # 2. Train Models
    logging.info(">>> [2/4] Training Ensemble...")
    
    model_dom = DOMINANT(input_dim).to(device)
    model_dom = train_model(model_dom, data, 'DOMINANT')
    s_dom = model_dom.get_anomaly_scores(data.x, data.edge_index).cpu().numpy()

    model_dae = AnomalyDAE(input_dim, num_nodes).to(device)
    model_dae = train_model(model_dae, data, 'AnomalyDAE')
    s_dae = model_dae.get_anomaly_scores(data.x, data.edge_index).cpu().numpy()

    model_cola = CoLA(input_dim).to(device)
    model_cola = train_model(model_cola, data, 'CoLA')
    s_cola = model_cola.get_anomaly_scores(data.x, data.edge_index).cpu().numpy()

    # 3. Aggregate
    logging.info(">>> [3/4] Aggregating Scores...")
    def norm(s): return (s - s.min()) / (s.max() - s.min() + 1e-8)
    
    final_score = (norm(s_dom) + norm(s_dae) + norm(s_cola)) / 3

    # 4. Save CSV Results
    logging.info(">>> [4/4] Saving Results...")
    mapping = pd.read_csv('/teamspace/uploads/cve_id_mapping.csv')
    num_cves = len(mapping)
    
    results = pd.DataFrame({
        'CVE_ID': mapping['CVE_ID'],
        'Final_Score': final_score[:num_cves],
        'DOM': s_dom[:num_cves],
        'DAE': s_dae[:num_cves],
        'CoLA': s_cola[:num_cves]
    })
    
    results = results.sort_values('Final_Score', ascending=False)
    results.to_csv('final_ensemble_results.csv', index=False)
    logging.info("✅ Saved CSV: final_ensemble_results.csv")

    # --- NEW: SAVE MODEL WEIGHTS ---
    logging.info(">>> Saving Model Weights for Phase 4 (Visualization)...")
    torch.save(model_dom.state_dict(), 'dominant_model.pt')
    torch.save(model_dae.state_dict(), 'anomalydae_model.pt')
    torch.save(model_cola.state_dict(), 'cola_model.pt')
    logging.info("✅ Saved Models: dominant_model.pt, anomalydae_model.pt, cola_model.pt")
    # -------------------------------

    logging.info("✅ DONE! Top 5 Anomalies:")
    print(results.head())

if __name__ == "__main__":
    main()

INFO: >>> [1/4] Loading Data...


INFO:    > Processing /teamspace/uploads/edges_cve_cwe.csv (232216 rows)...
INFO:      + Loaded 232216 edges.
INFO:    > Processing /teamspace/uploads/edges_cve_cpe.csv (1374447 rows)...
INFO:      + Loaded 1374447 edges.
INFO:    > Processing /teamspace/uploads/edges_cpe_product.csv (220433 rows)...
INFO:      + Loaded 220433 edges.
INFO:    > Processing /teamspace/uploads/edges_product_vendor.csv (89008 rows)...
INFO:      + Loaded 89008 edges.
INFO:    > Processing /teamspace/uploads/edges_cve_ref.csv (555033 rows)...
INFO:      + Loaded 555033 edges.
INFO:    > Removing duplicate edges (Coalescing)...
INFO:    > Adding noise features for 671410 auxiliary nodes (CWE/CPE/etc)...
INFO: ✅ FINAL GRAPH: 873161 Nodes, 4942274 Edges
INFO: >>> [2/4] Training Ensemble...
INFO: Training DOMINANT...


   Epoch 0: Loss 0.8013
   Epoch 10: Loss 0.6767
   Epoch 20: Loss 0.6248
   Epoch 30: Loss 0.5938
   Epoch 40: Loss 0.5746


INFO: Training AnomalyDAE...


   Epoch 0: Loss 0.3085
   Epoch 10: Loss 0.2374
   Epoch 20: Loss 0.1990
   Epoch 30: Loss 0.1821
   Epoch 40: Loss 0.1696


INFO: Training CoLA...


   Epoch 0: Loss 0.7097
   Epoch 10: Loss -0.0000
   Epoch 20: Loss -0.0000
   Epoch 30: Loss -0.0000
   Epoch 40: Loss -0.0000


INFO: >>> [3/4] Aggregating Scores...
INFO: >>> [4/4] Saving Results...
INFO: ✅ Saved CSV: final_ensemble_results.csv
INFO: >>> Saving Model Weights for Phase 4 (Visualization)...
INFO: ✅ Saved Models: dominant_model.pt, anomalydae_model.pt, cola_model.pt
INFO: ✅ DONE! Top 5 Anomalies:


                CVE_ID  Final_Score       DOM       DAE  CoLA
126809  CVE-2022-34776     0.520366  0.360623  1.273236   0.0
95865   CVE-2021-24167     0.517881  0.398353  1.147780   0.0
126148  CVE-2022-33989     0.516897  0.325490  1.363783   0.0
102139  CVE-2021-32937     0.513960  0.312401  1.390045   0.0
137621   CVE-2023-0772     0.502768  0.411708  1.038709   0.0
